# CDC-CP P1 round 3 on Kaggle

Three variants of the best P1 (backbone 1e-4, 20 epochs, Baseline 1 augmentation), each with seeds 42 and 43:

| Key | Config | Change from the best P1 |
|---|---|---|
| **W42**, **W43** | `configs/p1c_wide.yaml` | projection head 1280 → 512 → 128 |
| **D42**, **D43** | `configs/p1c_deep.yaml` | projection head 1280 → 512 → 256 → 128 |
| **C42**, **C43** | `configs/p1c_convnext.yaml` | ConvNeXt-Tiny backbone instead of EfficientNet-B0 |

The control is round 1's `p1_lr1e-4` seeds 42 and 43, which already exist.

**Setup (once)**
1. Add two Kaggle Datasets as inputs: the code (`cdc-cp_code.zip`) and the data (`DPD_Small_train.csv` plus the `DPD_Small_dataset` image folder, without the test CSVs).
2. Settings panel: **Accelerator → GPU T4 x2**, **Internet → On** (needed to download ImageNet weights; requires a verified phone number).

**To run everything unattended:** click **Save Version → Save & Run All (Commit)**. Kaggle runs the whole notebook in the background (up to 12 hours) and keeps everything written to `/kaggle/working` as the version's output. You can close the browser.

**If a version stops early** (time limit or error): open the notebook, **Add Input → Your Work → this notebook's last version** (its output), then Save & Run All again. Cell 4 copies the finished and half-finished runs back, finished runs are skipped, and the interrupted one resumes from its last finished epoch. Nothing needs editing.

**Rules this notebook keeps:** only `DPD_Small_train.csv` and the 8,000 images it lists are read; official test files are never opened. No early stopping.

## 1. Settings

In [ ]:
from pathlib import Path

RUN      = 'ROUND3'   # 'ROUND3' trains all six in a row, or one key: 'W42', 'W43', 'D42', 'D43', 'C42', 'C43'
WORKERS  = 4          # Kaggle gives 4 CPU cores
CHECKS   = False      # True runs the unit tests and a 60-step timing first

INPUT     = Path('/kaggle/input')
RUNS_DIR  = Path('/kaggle/working/runs')   # saved as the version's output
CODE_DIR  = Path('/tmp/cdc-cp')            # not saved
DATA_DIR  = Path('/tmp/DPD_Small')         # not saved
CACHE_DIR = Path('/tmp/cache')             # not saved

## 2. GPU check and code

In [ ]:
import os, shutil, subprocess, sys, zipfile
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__)

# The code dataset may appear as the zip itself or, if Kaggle unpacked it, as a cdc-cp folder.
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
found = next((p.parent.parent for p in INPUT.rglob('cdccp/__init__.py')), None)
if found is not None:
    shutil.copytree(found, CODE_DIR)
else:
    code_zip = next(INPUT.rglob('cdc-cp_code*.zip'), None)
    assert code_zip, 'Add the code dataset (cdc-cp_code.zip) as an input'
    with zipfile.ZipFile(code_zip) as z:
        z.extractall('/tmp')
assert (CODE_DIR / 'cdccp').is_dir(), f'code not found in {CODE_DIR}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
print('code from', found or code_zip)


def sh(*args):
    """Run a command in the repo and stream its output."""
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=dict(os.environ),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 3. Training data (train CSV and its 8,000 images only)
Nothing is copied: a local folder links to the input images.

In [ ]:
import hashlib
import pandas as pd

train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'

DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, DATA_DIR / 'DPD_Small_train.csv')
link = DATA_DIR / 'DPD_Small_dataset'
if not link.exists():
    link.symlink_to(image_src, target_is_directory=True)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

## 4. Restore runs from an earlier version's output
If you attached a previous version of this notebook as an input, its runs are copied back so training continues.

In [ ]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
restored = []
for last in INPUT.rglob('p1c_*_seed*/last.pt'):
    src, dst = last.parent, RUNS_DIR / last.parent.name
    old = dst / 'last.pt'
    if not old.exists() or torch.load(old, map_location='cpu', weights_only=False)['applied'] < torch.load(last, map_location='cpu', weights_only=False)['applied']:
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
        restored.append(dst.name)
print('restored:', restored or 'nothing (first version)')

## 5. Run list and shared overrides

In [ ]:
COMMON = [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']
RUNS = {
    'W42': ('configs/p1c_wide.yaml', []),     'W43': ('configs/p1c_wide.yaml', ['seed=43']),
    'D42': ('configs/p1c_deep.yaml', []),     'D43': ('configs/p1c_deep.yaml', ['seed=43']),
    'C42': ('configs/p1c_convnext.yaml', []), 'C43': ('configs/p1c_convnext.yaml', ['seed=43']),
}
ROUND3 = ['W42', 'W43', 'D42', 'D43', 'C42', 'C43']   # cheap head variants first, ConvNeXt last
TODO = ROUND3 if RUN == 'ROUND3' else [RUN]
print(TODO)

## 6. Build the image cache (local disk, a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 7. (Optional, CHECKS = True) Unit tests and timing

In [ ]:
if CHECKS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
    sh(sys.executable, '-m', 'pytest', '-q', 'tests/test_model_bn_loss.py', 'tests/test_evaluate.py', 'tests/test_train.py')
    for config in ('configs/p1c_wide.yaml', 'configs/p1c_convnext.yaml'):
        sh(sys.executable, 'scripts/time_run.py', '--config', config, '--workers', WORKERS, '--steps', 60, f'paths.cache_dir={CACHE_DIR}')
else:
    print('skipped (CHECKS = False)')

## 8. Train
Trains each run in `TODO` in turn. Finished runs print `nothing to do`; an interrupted run resumes from its last finished epoch.

In [ ]:
for key in TODO:
    config, extra = RUNS[key]
    print(f'===== {key}: {config} {" ".join(extra)} =====', flush=True)
    sh(sys.executable, 'scripts/train.py', config, *COMMON, *extra, '--resume')

## 9. Selection tables
One `selection.md` per run. A person reads and signs each.

In [ ]:
import yaml
for key in TODO:
    config, extra = RUNS[key]
    name = yaml.safe_load(open(CODE_DIR / config))['name']
    run_dir = RUNS_DIR / f"{name}_seed{43 if 'seed=43' in extra else 42}"
    sh(sys.executable, 'scripts/select_table.py', run_dir)
    print(open(run_dir / 'selection.md', encoding='utf-8').read())

## 10. Curves

In [ ]:
import matplotlib.pyplot as plt

runs = sorted(p for p in RUNS_DIR.iterdir() if (p / 'history.csv').is_file())
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for p in runs:
    h = pd.read_csv(p / 'history.csv')
    line, = axes[0].plot(h.epoch, h.A_S, label=f'{p.name}')
    axes[1].plot(h.epoch, h.unseen_disease_top1, color=line.get_color(), label=f'{p.name}')
    axes[2].plot(h.epoch, h.val_ce_disease, color=line.get_color(), label=f'{p.name}')
for ax, title in zip(axes, ['Seen pairs Top-1', 'Unseen disease Top-1', 'Validation disease cross-entropy (seen)']):
    ax.set_title(title); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 11. Round 3 comparison
Needs the control runs (`p1_lr1e-4_seed42` and `_seed43`). If they are not attached as an input, download this version's output and run `scripts/p1_compare.py` on your PC instead.

In [ ]:
for s in (42, 43):
    src = next((p.parent for p in INPUT.rglob(f'p1_lr1e-4_seed{s}/history.csv')), None)
    dst = RUNS_DIR / f'p1_lr1e-4_seed{s}'
    if src is not None and not dst.exists():
        dst.mkdir()
        for f in ('history.csv', 'last.pt'):
            if (src / f).exists():
                shutil.copy2(src / f, dst / f)
if all((RUNS_DIR / f'p1_lr1e-4_seed{s}' / 'last.pt').is_file() for s in (42, 43)):
    sh(sys.executable, 'scripts/p1_compare.py', RUNS_DIR, '--control', 'p1_lr1e-4',
       '--variants', 'p1c_wide', 'p1c_deep', 'p1c_convnext', '--out', RUNS_DIR / 'p1_round3.md')
else:
    print('Control runs not attached: compare on your PC after downloading the output.')